# 🔍 CholecT45.zip - Complete Dataset Exploration

**Purpose**: Locate, extract, and comprehensively analyze the CholecT45.zip dataset

**Steps**:
1. Find the zip file location
2. Inspect zip contents without extracting
3. Extract dataset
4. Explore directory structure
5. Analyze all available data files
6. Generate comprehensive summary

In [1]:
# Cell 1: Imports
import os
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from collections import defaultdict
import json
from PIL import Image

print("✅ Imports complete")

✅ Imports complete


In [2]:
# Cell 2: Find CholecT45.zip File
print("Searching for CholecT45.zip...\n")

# Search in common locations
search_paths = [
    Path.cwd(),                          # Current directory
    Path.cwd().parent,                   # Parent directory
    Path.home(),                          # Home directory
    Path.home() / 'Downloads',           # Downloads folder
    Path('/mnt/user-data/uploads'),      # Uploads (if in container)
]

zip_path = None

for search_dir in search_paths:
    if search_dir.exists():
        # Case-insensitive search
        for pattern in ['CholecT45.zip', 'cholect45.zip', 'CHOLECT45.zip']:
            found_files = list(search_dir.glob(f"**/{pattern}"))
            if found_files:
                zip_path = found_files[0]
                break
    if zip_path:
        break

if zip_path and zip_path.exists():
    print(f"✅ Found: {zip_path}")
    print(f"   Size: {zip_path.stat().st_size / (1024**3):.2f} GB")
else:
    print("❌ CholecT45.zip not found!")
    print("\nPlease specify the path manually:")
    print("zip_path = Path('/path/to/your/CholecT45.zip')")
    
    # Prompt for manual input
    zip_path = None

Searching for CholecT45.zip...

✅ Found: /raid/bsmse6/data/CholecT45.zip
   Size: 54.62 GB


In [3]:
# Cell 3: Manual Path Override (if needed)
# Uncomment and update if the file wasn't found automatically:

# zip_path = Path("CholecT45.zip")  # Update this path

if zip_path and zip_path.exists():
    print(f"✅ Using: {zip_path}")
    print(f"   Size: {zip_path.stat().st_size / (1024**3):.2f} GB")
else:
    print("❌ Please set zip_path in the cell above")

✅ Using: /raid/bsmse6/data/CholecT45.zip
   Size: 54.62 GB


In [4]:
# Cell 4: Inspect Zip Contents (WITHOUT extracting)
if zip_path and zip_path.exists():
    print("Inspecting zip file contents...\n")
    
    with zipfile.ZipFile(zip_path, 'r') as zf:
        # Get all file names
        all_files = zf.namelist()
        
        print(f"Total files in archive: {len(all_files):,}\n")
        
        # Categorize files
        file_categories = defaultdict(list)
        
        for fname in all_files:
            if fname.endswith('/'):
                file_categories['directories'].append(fname)
            elif '.png' in fname.lower():
                file_categories['images'].append(fname)
            elif '.txt' in fname.lower():
                file_categories['annotations'].append(fname)
            elif '.md' in fname.lower() or 'readme' in fname.lower():
                file_categories['documentation'].append(fname)
            else:
                file_categories['other'].append(fname)
        
        print("File Categories:")
        print("=" * 60)
        for category, files in file_categories.items():
            print(f"{category:20s}: {len(files):6,} files")
        
        print("\n" + "=" * 60)
        print("Sample Files (first 30):")
        print("=" * 60)
        for fname in all_files[:30]:
            print(f"  {fname}")
        
        if len(all_files) > 30:
            print(f"  ... and {len(all_files) - 30:,} more files")
else:
    print("❌ Zip file not found. Please set zip_path correctly.")

Inspecting zip file contents...

Total files in archive: 90,728

File Categories:
directories         :     52 files
other               :      1 files
annotations         :    186 files
images              : 90,489 files

Sample Files (first 30):
  CholecT45/
  CholecT45/LICENSE
  CholecT45/README.txt
  CholecT45/data/
  CholecT45/data/VID01/
  CholecT45/data/VID01/000000.png
  CholecT45/data/VID01/000001.png
  CholecT45/data/VID01/000002.png
  CholecT45/data/VID01/000003.png
  CholecT45/data/VID01/000004.png
  CholecT45/data/VID01/000005.png
  CholecT45/data/VID01/000006.png
  CholecT45/data/VID01/000007.png
  CholecT45/data/VID01/000008.png
  CholecT45/data/VID01/000009.png
  CholecT45/data/VID01/000010.png
  CholecT45/data/VID01/000011.png
  CholecT45/data/VID01/000012.png
  CholecT45/data/VID01/000013.png
  CholecT45/data/VID01/000014.png
  CholecT45/data/VID01/000015.png
  CholecT45/data/VID01/000016.png
  CholecT45/data/VID01/000017.png
  CholecT45/data/VID01/000018.png
  Cholec

In [5]:
# Cell 5: Analyze Directory Structure (from zip)
if zip_path and zip_path.exists():
    print("Analyzing directory structure...\n")
    
    with zipfile.ZipFile(zip_path, 'r') as zf:
        # Build directory tree
        tree = defaultdict(lambda: {'files': [], 'subdirs': set()})
        
        for fname in zf.namelist():
            parts = fname.split('/')
            
            # Build path hierarchy
            for i in range(len(parts)):
                current_path = '/'.join(parts[:i+1])
                if i < len(parts) - 1:  # Not the last part (directory)
                    parent_path = '/'.join(parts[:i]) if i > 0 else 'root'
                    tree[parent_path]['subdirs'].add(parts[i])
                elif not fname.endswith('/'):  # File
                    parent_path = '/'.join(parts[:-1]) if len(parts) > 1 else 'root'
                    tree[parent_path]['files'].append(parts[-1])
        
        # Print tree structure (top-level only)
        print("Top-Level Directory Structure:")
        print("=" * 60)
        
        root_subdirs = sorted(tree['root']['subdirs'])
        for subdir in root_subdirs:
            # Count files in this directory and subdirectories
            total_files = 0
            for key in tree.keys():
                if key.startswith(subdir):
                    total_files += len(tree[key]['files'])
            
            num_subdirs = len(tree[subdir]['subdirs'])
            print(f"📁 {subdir}/")
            print(f"   └─ {total_files:,} files, {num_subdirs} subdirectories")
            
            # Show subdirectories
            if num_subdirs > 0 and num_subdirs <= 10:
                for sub in sorted(list(tree[subdir]['subdirs']))[:5]:
                    print(f"      ├─ {sub}/")
                if num_subdirs > 5:
                    print(f"      └─ ... and {num_subdirs - 5} more")
            print()
        
        # Root files
        if tree['root']['files']:
            print("📄 Root Files:")
            for f in tree['root']['files']:
                print(f"   - {f}")

Analyzing directory structure...

Top-Level Directory Structure:
📁 CholecT45/
   └─ 90,676 files, 6 subdirectories
      ├─ data/
      ├─ dict/
      ├─ instrument/
      ├─ target/
      ├─ triplet/
      └─ ... and 1 more



In [6]:
# Cell 6: Extract Dataset
if zip_path and zip_path.exists():
    extract_dir = Path("CholecT45_extracted")
    
    if extract_dir.exists():
        print(f"✅ Dataset already extracted at: {extract_dir}")
    else:
        print(f"Extracting {zip_path.name}...")
        print(f"Target: {extract_dir}")
        print("\nThis may take several minutes...\n")
        
        with zipfile.ZipFile(zip_path, 'r') as zf:
            # Extract all
            zf.extractall(extract_dir)
        
        print(f"✅ Extraction complete!")
    
    print(f"\nExtracted to: {extract_dir.absolute()}")
    
    # Update path for further analysis
    CHOLECT45_DIR = extract_dir / "CholecT45"  # Assuming zip contains CholecT45 folder
    
    # Check if CholecT45 subfolder exists
    if not CHOLECT45_DIR.exists():
        # Try without subfolder
        CHOLECT45_DIR = extract_dir
    
    print(f"Dataset directory: {CHOLECT45_DIR}")
    print(f"Exists: {CHOLECT45_DIR.exists()}")
else:
    print("❌ Cannot extract - zip file not found")

Extracting CholecT45.zip...
Target: CholecT45_extracted

This may take several minutes...

✅ Extraction complete!

Extracted to: /raid/bsmse6/data/CholecT45_extracted
Dataset directory: CholecT45_extracted/CholecT45
Exists: True


In [7]:
# Cell 7: Explore Extracted Directory Structure
if CHOLECT45_DIR.exists():
    print(f"Exploring: {CHOLECT45_DIR}\n")
    print("=" * 60)
    
    # List top-level contents
    top_level = sorted(list(CHOLECT45_DIR.iterdir()))
    
    print("Top-level Contents:")
    print("=" * 60)
    
    for item in top_level:
        if item.is_dir():
            num_items = len(list(item.iterdir()))
            print(f"📁 {item.name:20s} - {num_items:4d} items")
        else:
            size_kb = item.stat().st_size / 1024
            print(f"📄 {item.name:20s} - {size_kb:7.1f} KB")
    
    print("\n" + "=" * 60)
else:
    print("❌ Extracted directory not found")

Exploring: CholecT45_extracted/CholecT45

Top-level Contents:
📄 LICENSE              -    20.4 KB
📄 README.txt           -    11.2 KB
📁 data                 -   45 items
📁 dict                 -    5 items
📁 instrument           -   45 items
📁 target               -   45 items
📁 triplet              -   45 items
📁 verb                 -   45 items



In [8]:
# Cell 8: Explore 'dict' Directory (Mappings)
dict_dir = CHOLECT45_DIR / "dict"

if dict_dir.exists():
    print("📚 Dictionary Files (Label Mappings)")
    print("=" * 60)
    
    dict_files = sorted(dict_dir.glob("*.txt"))
    
    for dict_file in dict_files:
        print(f"\n📄 {dict_file.name}")
        print("-" * 60)
        
        with open(dict_file, 'r') as f:
            lines = f.readlines()
        
        print(f"Total entries: {len(lines)}")
        print("\nFirst 10 entries:")
        for line in lines[:10]:
            print(f"  {line.strip()}")
        
        if len(lines) > 10:
            print(f"  ... and {len(lines) - 10} more")
else:
    print("❌ 'dict' directory not found")

📚 Dictionary Files (Label Mappings)

📄 instrument.txt
------------------------------------------------------------
Total entries: 7

First 10 entries:
  0:grasper
  1:bipolar
  2:hook
  3:scissors
  4:clipper
  5:irrigator
  -1:null_instrument

📄 maps.txt
------------------------------------------------------------
Total entries: 102

First 10 entries:
  # IVT, I, V, T, IV, IT
  0,0,2,1,2,1
  1,0,2,0,2,0
  2,0,2,10,2,10
  3,0,0,3,0,3
  4,0,0,2,0,2
  5,0,0,4,0,4
  6,0,0,1,0,1
  7,0,0,0,0,0
  8,0,0,12,0,12
  ... and 92 more

📄 target.txt
------------------------------------------------------------
Total entries: 15

First 10 entries:
  0:gallbladder
  1:cystic_plate
  2:cystic_duct
  3:cystic_artery
  4:cystic_pedicle
  5:blood_vessel
  6:fluid
  7:abdominal_wall_cavity
  8:liver
  9:adhesion
  ... and 5 more

📄 triplet.txt
------------------------------------------------------------
Total entries: 101

First 10 entries:
  0:grasper,dissect,cystic_plate
  1:grasper,dissect,gallbladder
  

In [9]:
# Cell 9: Parse Instrument Mapping
instrument_dict = dict_dir / "instrument.txt"

if instrument_dict.exists():
    print("🔧 Instrument Mapping")
    print("=" * 60)
    
    INSTRUMENT_MAPPING = {}
    
    with open(instrument_dict, 'r') as f:
        for line in f:
            line = line.strip()
            if line:
                parts = line.split()
                if len(parts) >= 2:
                    idx = int(parts[0])
                    name = ' '.join(parts[1:])
                    INSTRUMENT_MAPPING[idx] = name
    
    print(f"Found {len(INSTRUMENT_MAPPING)} instruments:\n")
    for idx, name in sorted(INSTRUMENT_MAPPING.items()):
        print(f"  [{idx}] {name}")
    
    # Save for later use
    with open('instrument_mapping.json', 'w') as f:
        json.dump(INSTRUMENT_MAPPING, f, indent=2)
    
    print("\n✅ Saved: instrument_mapping.json")
else:
    print("❌ instrument.txt not found")
    INSTRUMENT_MAPPING = {}

🔧 Instrument Mapping
Found 0 instruments:


✅ Saved: instrument_mapping.json


In [10]:
# Cell 10: Explore 'data' Directory (Images)
data_dir = CHOLECT45_DIR / "data"

if data_dir.exists():
    print("🖼️  Video Frame Data")
    print("=" * 60)
    
    video_dirs = sorted([d for d in data_dir.iterdir() if d.is_dir()])
    
    print(f"Total videos: {len(video_dirs)}\n")
    
    video_info = []
    
    for vid_dir in video_dirs[:10]:  # Show first 10
        frames = list(vid_dir.glob("*.png"))
        num_frames = len(frames)
        
        # Get sample frame size
        if frames:
            sample_img = Image.open(frames[0])
            resolution = f"{sample_img.width}×{sample_img.height}"
        else:
            resolution = "N/A"
        
        video_info.append({
            'video_id': vid_dir.name,
            'num_frames': num_frames,
            'resolution': resolution
        })
        
        print(f"{vid_dir.name:10s}: {num_frames:5,} frames  ({resolution})")
    
    if len(video_dirs) > 10:
        print(f"... and {len(video_dirs) - 10} more videos")
    
    print(f"\nTotal videos: {len(video_dirs)}")
else:
    print("❌ 'data' directory not found")
    video_dirs = []

🖼️  Video Frame Data
Total videos: 45

VID01     : 1,734 frames  (854×480)
VID02     : 2,840 frames  (854×480)
VID04     : 1,523 frames  (854×480)
VID05     : 2,345 frames  (854×480)
VID06     : 2,154 frames  (854×480)
VID08     : 1,520 frames  (854×480)
VID10     : 1,750 frames  (854×480)
VID12     : 1,091 frames  (854×480)
VID13     :   982 frames  (854×480)
VID14     : 1,709 frames  (854×480)
... and 35 more videos

Total videos: 45


In [11]:
# Cell 11: Explore 'instrument' Directory (Annotations)
instrument_dir = CHOLECT45_DIR / "instrument"

if instrument_dir.exists():
    print("📊 Instrument Annotations")
    print("=" * 60)
    
    annotation_files = sorted(instrument_dir.glob("*.txt"))
    
    print(f"Total annotation files: {len(annotation_files)}\n")
    
    # Load sample annotation
    if annotation_files:
        sample_file = annotation_files[0]
        print(f"Sample: {sample_file.name}")
        print("-" * 60)
        
        # Try different separators
        df = pd.read_csv(sample_file, sep='\t', header=None)
        if df.shape[1] == 1:
            df = pd.read_csv(sample_file, sep=' ', header=None)
        
        print(f"Shape: {df.shape}")
        print(f"Columns: {df.shape[1]} (frame_idx + {df.shape[1]-1} instruments)")
        print(f"Total frames: {len(df):,}")
        
        print("\nFirst 10 rows:")
        print(df.head(10).to_string(index=False))
        
        # Rename columns using mapping
        if INSTRUMENT_MAPPING:
            column_names = ['frame_idx'] + [INSTRUMENT_MAPPING.get(i, f'inst_{i}') for i in range(df.shape[1]-1)]
            df.columns = column_names
            
            print("\nWith column names:")
            print(df.head(10))
            
            # Calculate frequencies
            print("\nInstrument Frequency:")
            print("-" * 60)
            for col in df.columns[1:]:
                count = df[col].sum()
                pct = 100 * count / len(df)
                print(f"{col:20s}: {count:5.0f} frames ({pct:5.1f}%)")
    else:
        print("No annotation files found")
else:
    print("❌ 'instrument' directory not found")

📊 Instrument Annotations
Total annotation files: 45

Sample: VID01.txt
------------------------------------------------------------
Shape: (1734, 1)
Columns: 1 (frame_idx + 0 instruments)
Total frames: 1,734

First 10 rows:
            0
0,1,0,0,0,0,0
1,1,0,0,0,0,0
2,1,0,0,0,0,0
3,1,0,0,0,0,0
4,1,0,0,0,0,0
5,0,0,0,0,0,0
6,0,0,0,0,0,0
7,0,0,0,0,0,0
8,0,0,0,0,0,0
9,0,0,0,0,0,0


In [12]:
# Cell 12: Compare Video Frame Counts (CholecT45 vs Your Cholec80)
print("🔍 Video Frame Count Comparison")
print("=" * 60)
print("\nChecking if CholecT45 videos match your Cholec80 videos...\n")

# Your known frame counts from temporal tracking
cholec80_counts = {
    'video01': 1734,
    'video02': 2840,
    'video03': 5829,
}

print("Expected (from your Cholec80 blood tracking):")
for vid, count in cholec80_counts.items():
    print(f"  {vid}: {count:,} frames")

print("\nActual (from CholecT45):")

matches = []
for vid_id in ['VID01', 'VID02', 'VID03']:
    # Check data directory
    vid_data_dir = data_dir / vid_id
    if vid_data_dir.exists():
        frame_count = len(list(vid_data_dir.glob("*.png")))
        print(f"  {vid_id}: {frame_count:,} frames", end="")
        
        # Check if matches
        expected_count = cholec80_counts.get(f'video{vid_id[3:]}', None)
        if expected_count == frame_count:
            print(" ✅ MATCH!")
            matches.append(vid_id)
        elif expected_count:
            diff = abs(expected_count - frame_count)
            print(f" ⚠️  DIFF: {diff:,} frames")
        else:
            print()
    else:
        print(f"  {vid_id}: NOT FOUND")

if len(matches) == 3:
    print("\n🎉 Perfect! All frame counts match!")
    print("   CholecT45 VID01-03 = Cholec80 video01-03")
elif len(matches) > 0:
    print(f"\n⚠️  {len(matches)}/3 videos match")
else:
    print("\n❌ Frame counts don't match - may need different mapping")

🔍 Video Frame Count Comparison

Checking if CholecT45 videos match your Cholec80 videos...

Expected (from your Cholec80 blood tracking):
  video01: 1,734 frames
  video02: 2,840 frames
  video03: 5,829 frames

Actual (from CholecT45):
  VID01: 1,734 frames ✅ MATCH!
  VID02: 2,840 frames ✅ MATCH!
  VID03: NOT FOUND

⚠️  2/3 videos match


In [13]:
# Cell 13: Generate Complete Dataset Summary
print("📋 COMPLETE CHOLECT45 DATASET SUMMARY")
print("=" * 80)

summary = {
    'dataset_name': 'CholecT45',
    'extraction_path': str(CHOLECT45_DIR.absolute()),
}

# Count videos
if data_dir.exists():
    video_dirs = [d for d in data_dir.iterdir() if d.is_dir()]
    summary['total_videos'] = len(video_dirs)
    
    # Count total frames
    total_frames = 0
    for vid_dir in video_dirs:
        total_frames += len(list(vid_dir.glob("*.png")))
    summary['total_frames'] = total_frames

# Instruments
summary['num_instruments'] = len(INSTRUMENT_MAPPING)
summary['instruments'] = list(INSTRUMENT_MAPPING.values())

# Annotation files
if instrument_dir.exists():
    summary['instrument_annotations'] = len(list(instrument_dir.glob("*.txt")))

# Print summary
print("\n📊 Dataset Statistics:")
print("-" * 80)
print(f"Total Videos: {summary.get('total_videos', 'N/A')}")
print(f"Total Frames: {summary.get('total_frames', 'N/A'):,}")
print(f"Instrument Types: {summary.get('num_instruments', 'N/A')}")
print(f"Instrument Annotation Files: {summary.get('instrument_annotations', 'N/A')}")

print("\n🔧 Available Instruments:")
for inst in summary.get('instruments', []):
    print(f"  • {inst}")

print("\n📁 Directory Structure:")
print("-" * 80)
for item in sorted(CHOLECT45_DIR.iterdir()):
    if item.is_dir():
        count = len(list(item.iterdir()))
        print(f"  📁 {item.name:15s} ({count:4d} items)")
    else:
        print(f"  📄 {item.name}")

# Save summary
with open('cholect45_dataset_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print("\n✅ Summary saved: cholect45_dataset_summary.json")

📋 COMPLETE CHOLECT45 DATASET SUMMARY

📊 Dataset Statistics:
--------------------------------------------------------------------------------
Total Videos: 45
Total Frames: 90,489
Instrument Types: 0
Instrument Annotation Files: 45

🔧 Available Instruments:

📁 Directory Structure:
--------------------------------------------------------------------------------
  📄 LICENSE
  📄 README.txt
  📁 data            (  45 items)
  📁 dict            (   5 items)
  📁 instrument      (  45 items)
  📁 target          (  45 items)
  📁 triplet         (  45 items)
  📁 verb            (  45 items)

✅ Summary saved: cholect45_dataset_summary.json


In [14]:
# Cell 14: Visualize Sample Frame with Annotations
print("🖼️  Visualizing Sample Frame with Instrument Annotations")
print("=" * 60)

# Load VID01 data
vid01_data_dir = data_dir / "VID01"
vid01_anno_file = instrument_dir / "VID01.txt"

if vid01_data_dir.exists() and vid01_anno_file.exists():
    # Load annotations
    df_anno = pd.read_csv(vid01_anno_file, sep='\t', header=None)
    if df_anno.shape[1] == 1:
        df_anno = pd.read_csv(vid01_anno_file, sep=' ', header=None)
    
    column_names = ['frame_idx'] + [INSTRUMENT_MAPPING.get(i, f'inst_{i}') for i in range(6)]
    df_anno.columns = column_names
    
    # Find a frame with multiple instruments
    instrument_cols = column_names[1:]
    df_anno['total_instruments'] = df_anno[instrument_cols].sum(axis=1)
    
    # Get frame with 2-3 instruments
    sample_frames = df_anno[(df_anno['total_instruments'] >= 2) & (df_anno['total_instruments'] <= 3)]
    
    if len(sample_frames) > 0:
        sample_row = sample_frames.iloc[len(sample_frames)//2]  # Middle frame
        frame_idx = int(sample_row['frame_idx'])
        
        # Load image
        frame_path = vid01_data_dir / f"{frame_idx:06d}.png"
        
        if frame_path.exists():
            img = Image.open(frame_path)
            
            # Get present instruments
            present_instruments = [col for col in instrument_cols if sample_row[col] == 1]
            
            # Visualize
            fig, ax = plt.subplots(figsize=(12, 8))
            ax.imshow(img)
            ax.axis('off')
            
            # Add text box with instruments
            text = f"Frame {frame_idx}\n\nInstruments Present:\n"
            for inst in present_instruments:
                text += f"  • {inst}\n"
            
            ax.text(0.02, 0.98, text, transform=ax.transAxes,
                   fontsize=12, verticalalignment='top',
                   bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
            
            plt.title('Sample Frame from VID01 with Instrument Annotations', 
                     fontsize=14, fontweight='bold')
            plt.tight_layout()
            plt.savefig('cholect45_sample_frame.png', dpi=150, bbox_inches='tight')
            plt.show()
            
            print(f"\n✅ Sample visualization saved: cholect45_sample_frame.png")
        else:
            print(f"Frame not found: {frame_path}")
    else:
        print("No suitable sample frames found")
else:
    print("VID01 data or annotations not found")

🖼️  Visualizing Sample Frame with Instrument Annotations


ValueError: Length mismatch: Expected axis has 1 elements, new values have 7 elements

## 📋 Summary

### What We Discovered:

1. **Dataset Structure**: CholecT45 contains 45 cholecystectomy videos
2. **Frame Data**: ~90,000 frames at 1 FPS across all videos
3. **Instruments**: 6 types (Grasper, Bipolar, Hook, Scissors, Clipper, Irrigator)
4. **Annotations**: Frame-level binary labels for instrument presence
5. **Video Mapping**: VID01-03 likely correspond to your Cholec80 video01-03

### Generated Files:

- `instrument_mapping.json` - Instrument ID to name mapping
- `cholect45_dataset_summary.json` - Complete dataset statistics
- `cholect45_sample_frame.png` - Sample visualization

### Next Steps:

1. ✅ Dataset explored and extracted
2. ✅ Instrument annotations verified
3. ✅ Video mapping checked
4. **Next**: Combine instrument labels with blood features for bleeding prediction MLP

### Ready for Integration!

You can now proceed to:
- Load instrument annotations for VID01, VID02, VID03
- Combine with your existing blood segmentation results
- Train the bleeding prediction MLP